# dispatch-back-fn-from-recipe — faded example 2: Dispatch raises KeyError for an unregistered (func, argnum) pair

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `dispatch-back-fn-from-recipe`. The last cell reports your progress on the `Backprop: dispatch back fn from recipe` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: dispatch back fn from recipe` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dispatch-back-fn-from-recipe`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dispatch-back-fn-from-recipe"
DD_SUBTOPIC = "Backprop: dispatch back fn from recipe"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The dispatch lookup uses `back_funcs[(node.recipe.func, argnum)]` as a plain dict access. If the key is missing, a `KeyError` is raised naturally. This is the intended behavior: a missing back function is an error in the registry setup, and the raised exception surfaces that problem immediately rather than silently skipping the parent.

## Faded exercise 2

Complete `dispatch_back_fns_strict`. The function should work identically to the basic dispatch for registered keys. For an unregistered `(func, argnum)` pair, it should let the natural `KeyError` propagate (no try/except needed — just do the lookup directly).

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
def dispatch_back_fns_strict(node, back_funcs):
    results = []
    for argnum, parent in node.recipe.parents.items():
        back_fn = back_funcs[(node.recipe.func, argnum)]
        results.append((argnum, parent, back_fn))
    return results


def _test():
    from dataclasses import dataclass
    from typing import Callable

    @dataclass
    class Recipe:
        func: Callable
        parents: dict

    class FakeTensor:
        def __init__(self, name, recipe=None):
            self.name = name
            self.recipe = recipe

    def exp_fn(x): return x  # fake
    def exp_back(g, out, x): return g * out

    x = FakeTensor('x')
    y = FakeTensor('y', Recipe(func=exp_fn, parents={0: x}))

    # Should succeed when key is present
    bf_good = {(exp_fn, 0): exp_back}
    triples = dispatch_back_fns_strict(y, bf_good)
    assert len(triples) == 1
    assert triples[0][2] is exp_back

    # Should raise KeyError when key is missing
    bf_empty = {}
    try:
        dispatch_back_fns_strict(y, bf_empty)
        assert False, 'Expected KeyError'
    except KeyError:
        pass


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def dispatch_back_fns_strict(node, back_funcs):
    results = []
    for argnum, parent in node.recipe.parents.items():
        back_fn = back_funcs[(node.recipe.func, argnum)]
        results.append((argnum, parent, back_fn))
    return results
```
</details>